<a href="https://colab.research.google.com/github/nattynatasha-gif/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 **Duka Electronics Sales Exploration**

**Half-year review (January–June 2026) for the Head of Sales.**

**Executive Summary**

- Nairobi CBD generated the highest revenue at **KES 25,855,067**, accounting for roughly 28% of the half-year total of KES 93.2 million.
- The rankings of branches by revenue and by number of orders are identical, with Nairobi CBD (277 orders) leading and Nakuru (101 orders) last.
- The top three products by revenue are the iPhone 13 (KES 13.5 m), Lenovo IdeaPad 3 (KES 11.6 m) and HP Pavilion 15 (KES 10.8 m); by units the leaders differ because lower-priced items move higher volumes.
- Weekend orders (279) deliver a higher average order revenue (KES 96,590) than weekday orders (KES 91,896), even though weekdays contribute more total volume.
- February was the best month for revenue (KES 16.8 m) while May was the weakest (KES 14.1 m), a percentage difference of 19.2%.

**First-five-minutes inspection**

 **Shape**: 1 000 rows × 8 columns.
- **Types**: order_date is datetime; quantity and unit_price are integers; rest are strings.
- **Missing values**: none.
- **Date range**: 2026-01-01 to 2026-06-30.
- **Key categories**: 6 branches, 14 products, 2 channels (In-store / Online), 5 cities.

 **Load data**

In [1]:
import os
import pandas as pd
import numpy as np

# Safely clean and rewrite the generator script if it exists
generator_path = "/content/generate_data.py"
if os.path.exists(generator_path):
    with open(generator_path, "r") as f:
        lines = f.readlines()

    # Skip lines until we find actual python code (like imports)
    clean_lines = []
    found_code = False
    for line in lines:
        if line.startswith("import ") or line.startswith("from "):
            found_code = True
        if found_code:
            clean_lines.append(line)

    if clean_lines:
        with open(generator_path, "w") as f:
            f.writelines(clean_lines)
        print("Successfully cleaned the generator script.")

    # Run the cleaned generator script to yield 'duka_sales.csv'
    %run /content/generate_data.py
else:
    print("Generator not found at path, dynamic fallback script simulation...")

# Load and inspect the generated sales dataset
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

# Create derived columns immediately so they are available globally
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

print("Shape:", df.shape)
print("\nDatatypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDate range:", df["order_date"].min().date(), "→", df["order_date"].max().date())
print("\nBranch value counts:")
print(df["branch"].value_counts())
print("\nChannel value counts:")
print(df["channel"].value_counts())

Successfully cleaned the generator script.
Wrote duka_sales.csv (1000 rows)
Wrote agents.csv (20 rows) and agent_transactions.csv (612 rows)
Shape: (1000, 13)

Datatypes:
order_id              object
order_date    datetime64[ns]
branch                object
city                  object
product               object
quantity               int64
unit_price             int64
channel               object
revenue                int64
month                  int32
weekday               object
is_weekend              bool
price_band            object
dtype: object

Missing values:
order_id      0
order_date    0
branch        0
city          0
product       0
quantity      0
unit_price    0
channel       0
revenue       0
month         0
weekday       0
is_weekend    0
price_band    0
dtype: int64

Date range: 2026-01-01 → 2026-06-30

Branch value counts:
branch
Nairobi CBD    277
Westlands      211
Mombasa        162
Kisumu         134
Eldoret        115
Nakuru         101
Name: count, dtype: 

 **Derived columns**

In [2]:
# Read the current content of the file
with open('/content/generate_data.py', 'r') as file:
    content = file.read()

# Find where the Python imports start, and skip the malformed header entirely
import_idx = content.find('import numpy as np')
if import_idx != -1:
    clean_code = content[import_idx:]
else:
    # Fallback search if spacing is different
    import_idx = content.find('import')
    clean_code = content[import_idx:] if import_idx != -1 else content

# Write the completely clean code back to the script
with open('/content/generate_data.py', 'w') as file:
    file.write(clean_code)

# Run the cleaned generator script to generate duka_sales.csv
%run /content/generate_data.py

# Re-load data and apply transformations
import pandas as pd
import numpy as np
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")
df[["order_id", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

Wrote duka_sales.csv (1000 rows)
Wrote agents.csv (20 rows) and agent_transactions.csv (612 rows)


,order_id,revenue,month,weekday,is_weekend,price_band
0,ORD00001,143652,6,Thursday,False,Premium
1,ORD00002,11787,2,Monday,False,Value
2,ORD00003,196148,1,Monday,False,Premium
3,ORD00004,89080,4,Sunday,True,Value
4,ORD00005,130221,3,Sunday,True,Premium


**Task 3**– Rank branches by total revenue and by number of orders

In [3]:
# Ensure derived columns exist on df
if "revenue" not in df.columns:
    df["revenue"] = df["quantity"] * df["unit_price"]
if "month" not in df.columns:
    df["month"] = df["order_date"].dt.month
if "weekday" not in df.columns:
    df["weekday"] = df["order_date"].dt.day_name()
if "is_weekend" not in df.columns:
    df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
if "price_band" not in df.columns:
    df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

# Re-rank branches by total revenue and order counts using the preloaded dataframe
rev_rank = df.groupby("branch")["revenue"].sum().sort_values(ascending=False)
ord_rank = df.groupby("branch").size().sort_values(ascending=False)
print("=== Revenue ranking ===")
print(rev_rank)
print("\n=== Order-count ranking ===")
print(ord_rank)
print("\nSame order?", list(rev_rank.index) == list(ord_rank.index))

=== Revenue ranking ===
branch
Nairobi CBD    25855067
Westlands      18698885
Mombasa        15268267
Kisumu         12781867
Eldoret        10771938
Nakuru          9829511
Name: revenue, dtype: int64

=== Order-count ranking ===
branch
Nairobi CBD    277
Westlands      211
Mombasa        162
Kisumu         134
Eldoret        115
Nakuru         101
dtype: int64

Same order? True


**Answer:** The rankings are identical. Nairobi CBD leads both lists and Nakuru is last. Average order values do not differ dramatically across branches, so the branch with the most orders also generates the most revenue.

 **Task 4** – Top three products by revenue and by units sold

In [4]:
print("Top 3 by revenue:")
print(df.groupby("product")["revenue"].sum().sort_values(ascending=False).head(3))
print("\nTop 3 by units:")
print(df.groupby("product")["quantity"].sum().sort_values(ascending=False).head(3))

Top 3 by revenue:
product
iPhone 13           13541182
Lenovo IdeaPad 3    11618748
HP Pavilion 15      10827881
Name: revenue, dtype: int64

Top 3 by units:
product
Canon EOS 2000D        219
LG 43-inch Smart TV    216
Lenovo IdeaPad 3       212
Name: quantity, dtype: int64


 **Task 5** – Weekend vs weekday comparison

In [5]:
print(df.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean")
).round(0))

            orders  total_revenue  avg_order_revenue
is_weekend                                          
False          721       66256946            91896.0
True           279       26948589            96590.0


**Answer:** Weekdays produce more orders and more total revenue. Weekend average order value is higher, suggesting weekend shoppers buy higher-value baskets.

**Task 6** – Best and worst month for revenue


In [6]:
mrev = df.groupby("month")["revenue"].sum()
print("=== Monthly Revenue ===")
print(mrev)
best, worst = mrev.idxmax(), mrev.idxmin()
pct = (mrev[best] - mrev[worst]) / mrev[worst] * 100
print(f"\nBest month: {best} (KES {mrev[best]:,.0f})")
print(f"Worst month: {worst} (KES {mrev[worst]:,.0f})")
print(f"Percentage difference: {pct:.1f}%")

=== Monthly Revenue ===
month
1    16184935
2    16807197
3    16616894
4    15307010
5    14099397
6    14190102
Name: revenue, dtype: int64

Best month: 2 (KES 16,807,197)
Worst month: 5 (KES 14,099,397)
Percentage difference: 19.2%


**Answer:** February was the strongest month and May the weakest. The percentage difference is 19.2%.

**Task 7** – Premium Online orders from Nairobi & Mombasa

In [7]:
prem = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["city"].isin(["Nairobi", "Mombasa"]))
]
print(f"Count: {len(prem)}")
print(f"Total revenue: KES {prem["revenue"].sum():,.0f}")

import os
os.makedirs("outputs", exist_ok=True)
prem.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Exported → outputs/premium_online_coast_capital.csv")

Count: 111
Total revenue: KES 13,471,660
Exported → outputs/premium_online_coast_capital.csv


**Answer:** There are 111 Premium Online orders from Nairobi and Mombasa with a combined revenue of KES 13,471,660.

**Create the Excel file**

In [8]:
import os
import pandas as pd

os.makedirs("excel", exist_ok=True)

branch_rev = df.groupby("branch")["revenue"].sum().reset_index()
branch_rev.columns = ["Branch", "Total_Revenue"]
branch_rev = branch_rev.sort_values("Total_Revenue", ascending=False)

region_map = {
    "Nairobi CBD": "Capital", "Westlands": "Capital", "Mombasa": "Coast",
    "Kisumu": "Western", "Nakuru": "Rift Valley", "Eldoret": "Rift Valley",
}
branch_rev["Region"] = branch_rev["Branch"].map(region_map)

pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum", fill_value=0)
pivot["Total"] = pivot.sum(axis=1)
pivot = pivot.reset_index()

with pd.ExcelWriter("excel/duka_branch_summary.xlsx", engine="openpyxl") as writer:
    branch_rev.to_excel(writer, sheet_name="Branch_Revenue_SUMIFS", index=False)
    pd.DataFrame(list(region_map.items()), columns=["Branch", "Region"]).to_excel(
        writer, sheet_name="Region_Lookup", index=False)
    pivot.to_excel(writer, sheet_name="Pivot_Branch_Channel", index=False)

print("Excel written")
print("Pandas total:", df["revenue"].sum())
print("Excel pivot total:", pivot["Total"].sum())
print("Match:", df["revenue"].sum() == pivot["Total"].sum())

Excel written
Pandas total: 93205535
Excel pivot total: 93205535
Match: True
